# Structured Output

##### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in. subsequent processing. LangChain supports multiple schema types and mehtods for enforcing structured output

## 1. Pydantic

##### Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [27]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq')
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x117846fd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1178479d0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This is the year of release of the movie")
    director:str=Field(description="This is the director of the movie")
    rating:float=Field(description="This is the imdb rating of the movie")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x110a51400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x110a52120>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie

In [6]:
response = model.invoke("details abt the movie interstellar")
print(response.content)

**Interstellar (2014)** – A science‑fiction epic that blends emotional storytelling with cutting‑edge physics.

| Item | Details |
|------|---------|
| **Director** | Christopher Nolan |
| **Screenplay** | Christopher Nolan (based on a story by Nolan & his brother Jonathan) |
| **Producers** | Christopher Nolan, Emma Thomas, Steven Spielberg, Kathleen Kennedy |
| **Main Cast** | • *Matthew McConaughey* as **Joseph Cooper** – a former NASA pilot turned farmer.<br>• *Anne Hathaway* as **Amelia Brand** – a brilliant scientist and astronaut.<br>• *Jessica Chastain* as **Murph** – Cooper’s daughter, grown‑up and a physicist.<br>• *Michael Caine* as **Professor Brand** – Amelia’s father, the mission’s mastermind.<br>• *Bill Irwin* as **Tom Cooper** – Murph’s father.<br>• *Matt Damon* as **Tom** – Cooper’s best friend, a fellow farmer. |
| **Genre** | Science‑fiction, Drama, Adventure |
| **Runtime** | 2 h 49 min |
| **Release Dates** | • U.S. (IMAX): 5 November 2014<br>• UK: 14 December 2014

In [ ]:
model_with_structure.invoke("provide details about the movie interstellar")


Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Tom Berenger', role='Barton')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

## TypeDict

#### TypeDict provides a simpler alternative using python's built-in typing, ideal when you dont need runtime validation

In [18]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with detailw"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's imdb rating"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("provide the details of the movie Obsession")
response


{'director': 'Gérard de Villiers',
 'rating': 7.6,
 'title': 'Obsession',
 'year': 2004}

In [24]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [26]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses

##### A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @Dataclass decorator

In [28]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""

    name: str = Field(description="The Name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    response_format = ContactInfo #auto selects providerstrategy
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])


name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [29]:
## dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""

    name: str
    email: str
    phone: str

agent = create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    response_format = ContactInfo

)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
